# Spurious Edge Correction
Loads the fullmatch edge CSV, detects pseudo-self-loops (edges where source and target share a name component and the weight is a statistical outlier), and replaces their weight with the exact full-match count. Outputs a corrected CSV ready for downstream analysis.

In [ ]:
import pandas as pd
import sys
import os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import filters, edge_processing
INPUT_DATASET = '../data/out/SpotlightWeightSource_0102_0505_fullmatch.csv'  # Update if needed!
DB_PATH = '../data/out/graph.db'  # Update if needed!

In [ ]:
import time
print(f"Loading and filtering: {INPUT_DATASET}")
start_time = time.time()
filtered_edges = edge_processing.read_edges_csv_and_filter_spurious(
    csv_path=INPUT_DATASET,
    db_path=DB_PATH
)
print(f"Done in {time.time() - start_time:.2f} seconds.")

In [ ]:
def calculate_final_weight(row):
    if row['z_score'] > 1.5 and row['has_shared_word']:
        if row['ordered_substring']:
            return 1.0
        else:
            return row['fullmatch_count']
    else:
        return row['weight']

filtered_edges['final_weight'] = filtered_edges.apply(calculate_final_weight, axis=1)

In [ ]:
base_name = os.path.splitext(os.path.basename(INPUT_DATASET))[0]
output_path = f'../data/out/{base_name}_corrected.csv'

output = filtered_edges.drop(columns=['has_shared_word', 'ordered_substring', 'z_score', 'fullmatch_count', 'final_weight'], errors='ignore').copy()
output['weight'] = filtered_edges['final_weight']
output.to_csv(output_path, index=False)
print(f"Saved corrected edges to {output_path}")